<a href="https://colab.research.google.com/github/varun4010/Customer_churn_project/blob/main/CustomerLeavingStop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import joblib
import kagglehub
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

#  load data
path = kagglehub.dataset_download("radheshyamkollipara/bank-customer-churn")
df = pd.read_csv(os.path.join(path, "Customer-Churn-Records.csv"))
df.drop(["Complain", "RowNumber", "CustomerId", "Surname"], axis=1, inplace=True)

y = df["Exited"]
df = df.drop("Exited", axis=1)

# feature engineering
df["ZeroBalance"] = (df["Balance"] == 0).astype(int)
df["BalanceSalaryRatio"] = df["Balance"] / (df["EstimatedSalary"] + 1)
df["AgeBucket"] = pd.cut(
    df["Age"], bins=[17, 30, 40, 50, 60, 100],
    labels=["18-30", "31-40", "41-50", "51-60", "60+"]
).astype(str)
df["TenureAgeRatio"] = df["Tenure"] / (df["Age"] + 1)
df["ProductsPerTenure"] = df["NumOfProducts"] / (df["Tenure"] + 1)

# ~4% of customers have Tenure=0, basically brand new signups -> flag it same as ZeroBalance
df["ZeroTenure"] = (df["Tenure"] == 0).astype(int)

cat_features = ["Geography", "Gender", "AgeBucket"]  # Card Type handled separately below, it's ordinal

CARD_TYPE_ORDER = {"SILVER": 0, "GOLD": 1, "PLATINUM": 2, "DIAMOND": 3}

#  train/val/test split
xtr_full, xts, ytr_full, yts = train_test_split(
    df, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
xtr, xval, ytr, yval = train_test_split(
    xtr_full, ytr_full, test_size=0.2, stratify=ytr_full, random_state=RANDOM_STATE
)

num_neg = np.sum(ytr == 0)
num_pos = np.sum(ytr == 1)
estimated_weight = num_neg / num_pos


def preprocess(X):
    # ordinal encode card type instead of treating it as a plain category
    X = X.copy()
    X["CardTypeOrdinal"] = X["Card Type"].map(CARD_TYPE_ORDER)
    X = X.drop(columns=["Card Type"])

    # fillna before casting to str, otherwise real NaNs turn into the string "nan"
    for c in cat_features:
        X[c] = X[c].fillna("Missing").astype(str)

    return X


preprocessor = FunctionTransformer(preprocess)

# optuna tuning, only on xtr/ytr via CV, xval/xts untouched here
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def objective(trial):
    cb_params = {
        "iterations": trial.suggest_categorical("iterations", [300, 500, 800, 1200]),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 3, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_categorical("border_count", [32, 64, 128, 254]),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, estimated_weight * 2, log=True),
        "loss_function": "Logloss",
        "eval_metric": "F1",
        "random_state": RANDOM_STATE,
        "verbose": False,
        "allow_writing_files": False,
        "thread_count": -1,
    }

    scores = []
    for train_idx, val_idx in cv.split(xtr, ytr):
        X_train_cv, X_val_cv = xtr.iloc[train_idx], xtr.iloc[val_idx]
        y_train_cv, y_val_cv = ytr.iloc[train_idx], ytr.iloc[val_idx]

        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("catboost", CatBoostClassifier(cat_features=cat_features, **cb_params)),
        ])
        pipe.fit(X_train_cv, y_train_cv)
        preds = pipe.predict(X_val_cv)
        scores.append(f1_score(y_val_cv, preds))

    return np.mean(scores)


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=RANDOM_STATE))

print("Starting Optuna optimization (CatBoost)...")
study.optimize(objective, n_trials=60, show_progress_bar=True)

print(f"Best CV F1: {study.best_value:.4f}")
print("Best params:", study.best_params)

# fit final model on full training set
best_params = study.best_params.copy()
best_params.update({
    "loss_function": "Logloss",
    "eval_metric": "F1",
    "random_state": RANDOM_STATE,
    "verbose": False,
    "allow_writing_files": False,
    "thread_count": -1,
})

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("catboost", CatBoostClassifier(cat_features=cat_features, **best_params)),
])
final_pipeline.fit(xtr, ytr)

# check on validation set first, before touching test set
yval_pred = final_pipeline.predict(xval)
print(f"\nValidation F1: {f1_score(yval, yval_pred):.4f}")

# final test set evaluation, only run once
# F1 is the sole evaluation metric per competition rules -> plain .predict()
# at the sklearn default 0.5 threshold, no predict_proba / custom threshold involved
ypr = final_pipeline.predict(xts)

print(f"\nTest F1: {f1_score(yts, ypr):.4f}")
print(classification_report(yts, ypr, target_names=["Stayed (0)", "Churned (1)"]))

joblib.dump({"pipeline": final_pipeline}, "bank_churn_catboost_pipeline.joblib")
print("Saved pipeline to bank_churn_catboost_pipeline.joblib")

Using Colab cache for faster access to the 'bank-customer-churn' dataset.
Starting Optuna optimization (CatBoost)...


  0%|          | 0/60 [00:00<?, ?it/s]

Best CV F1: 0.6282
Best params: {'iterations': 800, 'learning_rate': 0.010211753133264504, 'depth': 6, 'l2_leaf_reg': 1.799629442281571, 'random_strength': 1.3495198637209393, 'bagging_temperature': 0.8000920381375523, 'border_count': 64, 'scale_pos_weight': 2.6341357526370577}

Validation F1: 0.6107

Test F1: 0.6421
              precision    recall  f1-score   support

  Stayed (0)       0.92      0.89      0.90      1592
 Churned (1)       0.61      0.68      0.64       408

    accuracy                           0.84      2000
   macro avg       0.76      0.78      0.77      2000
weighted avg       0.85      0.84      0.85      2000

Saved pipeline to bank_churn_catboost_pipeline.joblib


In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.3 MB/s eta 0:00:00


In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 22.1 MB/s eta 0:00:00


In [6]:
pip install ydata_profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.5/682.5 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.0 MB/s eta 0:00:00


In [7]:
import os
import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split
from ydata_profiling import ProfileReport

RANDOM_STATE = 42


path = kagglehub.dataset_download("radheshyamkollipara/bank-customer-churn")
r = os.path.join(path, 'Customer-Churn-Records.csv')
df = pd.read_csv(r)
df.drop(['Complain', 'RowNumber', 'CustomerId', 'Surname'], inplace=True, axis=1)

y = df['Exited']
X = df.drop('Exited', axis=1)

xtr_full, xts, ytr_full, yts = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
xtr, xval, ytr, yval = train_test_split(
    xtr_full, ytr_full, test_size=0.2, stratify=ytr_full, random_state=RANDOM_STATE
)

eda_df = xtr.copy()
eda_df['Exited'] = ytr.values

profile = ProfileReport(
    eda_df,
    title="Bank Customer Churn — Train Split EDA",
    explorative=True,
)
profile.to_file("output_train_eda.html")

print(f"Profiled {len(eda_df)} training rows only "
      f"({len(xval)} val / {len(xts)} test rows excluded from this report).")

/tmp/ipykernel_22675/2209335090.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Using Colab cache for faster access to the 'bank-customer-churn' dataset.


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 14/14 [00:00<00:00, 39.61it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profiled 6400 training rows only (1600 val / 2000 test rows excluded from this report).


In [ ]:

import kagglehub
import os
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix,classification_report
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
path = kagglehub.dataset_download("radheshyamkollipara/bank-customer-churn")
r=os.path.join(path,'Customer-Churn-Records.csv')
df=pd.read_csv(r)
df.drop(['Complain','RowNumber','CustomerId','Surname'],inplace=True,axis=1)
y=df['Exited']
df=df.drop(['Exited'],axis=1)
t=ColumnTransformer(transformers=[('t1',StandardScaler(),['CreditScore','Age','Tenure','Balance','NumOfProducts','EstimatedSalary','Point Earned']),('t2',OneHotEncoder(drop='first',sparse_output=False),['Geography','Gender']),('t3',OrdinalEncoder(categories=[['SILVER', 'GOLD', 'PLATINUM', 'DIAMOND']]),['Card Type'])],remainder='passthrough')
xtr,xts,ytr,yts=train_test_split(df,y,test_size=0.2,random_state=42)
xtr=t.fit_transform(xtr)
xts=t.transform(xts)
pr=DecisionTreeClassifier(max_depth=8 ,min_samples_leaf=5)
pr.fit(xtr,ytr)
ypr=pr.predict(xts)
d=classification_report(ypr,yts)
print(d)



Using Colab cache for faster access to the 'bank-customer-churn' dataset.
              precision    recall  f1-score   support

           0       0.94      0.88      0.91      1727
           1       0.47      0.67      0.55       273

    accuracy                           0.85      2000
   macro avg       0.70      0.77      0.73      2000
weighted avg       0.88      0.85      0.86      2000



In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.8 MB/s eta 0:00:00


In [ ]:

import kagglehub
import os
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix,classification_report
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
path = kagglehub.dataset_download("radheshyamkollipara/bank-customer-churn")
r=os.path.join(path,'Customer-Churn-Records.csv')
df=pd.read_csv(r)
df.drop(['Complain','RowNumber','CustomerId','Surname'],inplace=True,axis=1)
y=df['Exited']

df.sample()

Using Colab cache for faster access to the 'bank-customer-churn' dataset.


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Satisfaction Score,Card Type,Point Earned
3134,694,France,Male,34,5,127900.03,1,1,0,101737.8,0,5,GOLD,325


In [ ]:

import kagglehub
import os
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, average_precision_score
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix,classification_report
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
path = kagglehub.dataset_download("radheshyamkollipara/bank-customer-churn")
r=os.path.join(path,'Customer-Churn-Records.csv')
df=pd.read_csv(r)
df.drop(['Complain','RowNumber','CustomerId','Surname'],inplace=True,axis=1)
y=df['Exited']
df=df.drop(['Exited'],axis=1)
t=ColumnTransformer(transformers=[('t1',StandardScaler(),['CreditScore','Age','Tenure','Balance','NumOfProducts','EstimatedSalary','Point Earned']),('t2',OneHotEncoder(drop='first',sparse_output=False),['Geography','Gender']),('t3',OrdinalEncoder(categories=[['SILVER', 'GOLD', 'PLATINUM', 'DIAMOND']]),['Card Type'])],remainder='passthrough')
xtr,xts,ytr,yts=train_test_split(df,y,test_size=0.2,random_state=42)
num_neg = np.sum(ytr==0)
num_pos = np.sum(ytr==1)
estimated_weight = num_neg / num_pos
xtr=t.fit_transform(xtr)
xts=t.transform(xts)
ic = xgb.XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    scale_pos_weight=estimated_weight,
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=42
)
ic.fit(xtr,ytr)
ypr=ic.predict(xts)

y_proba = ic.predict_proba(xts)[:, 1]
print(f"PR-AUC (Precision-Recall Curve): {average_precision_score(yts, y_proba):.4f}\n")
print(classification_report(yts, ypr, target_names=["Majority (0)", "Minority (1)"]))



Using Colab cache for faster access to the 'bank-customer-churn' dataset.
PR-AUC (Precision-Recall Curve): 0.7042

              precision    recall  f1-score   support

Majority (0)       0.94      0.82      0.88      1607
Minority (1)       0.52      0.78      0.62       393

    accuracy                           0.81      2000
   macro avg       0.73      0.80      0.75      2000
weighted avg       0.86      0.81      0.83      2000



In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 18.8 MB/s eta 0:00:00


In [ ]:
import joblib

In [ ]:
!pip install catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 10.7 MB/s eta 0:00:00
